# Task 1 — Train BERT4Rec (time-split 80/20)

Giữ nguyên kiến trúc BERT4Rec gốc, chia lại data theo thời gian (20% cuối mỗi user = test). Không leakage. Log wandb.

In [1]:
import os, sys, json, torch
import torch.optim as optim
# path: chạy được dù cwd ở repo-root hay trong longterm_rerank
HERE = os.path.abspath("longterm_rerank") if os.path.isdir("longterm_rerank") else os.getcwd()
sys.path.insert(0, HERE)
import common, run_pipeline as R
from common import set_seed, load_and_split, BERT4Rec, BERT4RecDataset
set_seed(42)
DEVICE = R.DEVICE
MAX_LEN = R.MAX_LEN
print("Device:", DEVICE, "| HERE:", HERE)
from Bert4Rec_model import train_one_epoch
import wandb

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.


wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\TanPhat\_netrc


wandb: Currently logged in as: lamgiang (lamgiang-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda | HERE: F:\1_REL\Reinforcement-learning-for-Recommendation\longterm_rerank


## 1. Chia dữ liệu (time-split) + thống kê

In [2]:
users, bert_seqs, n_real, movie2id, stats = load_and_split(R.RATING_FILE, max_len=MAX_LEN)
json.dump(stats, open(os.path.join(HERE,"data_stats.json"),"w"), ensure_ascii=False, indent=2)
print(json.dumps(stats, ensure_ascii=False, indent=2))
n_model = n_real + 1   # mask = n_real+1 (khớp convention BERT4Rec)

{
  "n_users": 6040,
  "n_items": 3706,
  "n_interactions": 1000209,
  "n_train_remaining": 800193,
  "n_test": 200016,
  "avg_seq_len": 165.6,
  "avg_test_len": 33.12,
  "test_frac": 0.2,
  "max_len": 200
}


## 2. Khởi tạo BERT4Rec (kiến trúc gốc) + Dataset Cloze

In [3]:
bert = BERT4Rec(n_items=n_model, max_seq_length=MAX_LEN, hidden_size=64,
                n_layers=2, n_heads=2, hidden_dropout_prob=0.2,
                attn_dropout_prob=0.2, loss_type="CE").to(DEVICE)
ds = BERT4RecDataset(bert_seqs, n_items=n_real, max_seq_len=MAX_LEN, mask_token=n_model, mask_ratio=0.2)
dl = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=True, drop_last=True)
opt = optim.Adam(bert.parameters(), lr=1e-4)
print("params:", sum(p.numel() for p in bert.parameters()))

params: 358203


## 3. Train + early-stop theo VAL graded NDCG@10 (+ wandb)

In [4]:
key = common.load_env_key(os.path.join(os.path.dirname(HERE), ".env"))
run = None
try:
    if key: wandb.login(key=key)
    run = wandb.init(project="bert4rec-longterm-rerank", name="nb-bert-retrain", reinit=True)
except Exception as e:
    print("wandb off:", e)

EP, PAT = 80, 10
val_sub = list(range(0, len(users), max(1, len(users)//2000)))[:2000]
best, best_sd, noimp = -1.0, None, 0
for ep in range(EP):
    loss, acc = train_one_epoch(bert, dl, opt, DEVICE)
    vnd = R.eval_bert(bert, users, DEVICE, ks=(10,), subset=val_sub)["NDCG@10"]
    tag = "  <- best" if vnd > best+1e-5 else f" (best {best:.4f},{noimp+1}/{PAT})"
    print(f"ep {ep+1:02d}/{EP}  loss={loss:.4f} acc={acc:.2f}%  VAL NDCG@10={vnd:.4f}{tag}")
    if run:
        try: wandb.log({"bert/loss":loss,"bert/acc":acc,"bert/val_ndcg10":vnd,"bert/epoch":ep+1})
        except Exception: pass
    if vnd > best+1e-5:
        best=vnd; best_sd={k:v.detach().clone() for k,v in bert.state_dict().items()}; noimp=0
    else:
        noimp+=1
        if noimp>=PAT: print(f"EARLY STOP ep {ep+1}, best VAL NDCG@10={best:.4f}"); break
if best_sd: bert.load_state_dict(best_sd)
torch.save(bert.state_dict(), os.path.join(HERE,"bert4rec_longterm.pth"))
if run:
    try: wandb.finish()
    except Exception: pass
print("saved bert4rec_longterm.pth | best VAL NDCG@10 =", round(best,4))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.


wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\TanPhat\_netrc


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: setting up run rv4j9phm


...

wandb: uploading config.yaml; uploading output.log


wandb: 
wandb: Run history:
wandb:        bert/acc ▁▂▂▂▂▃▅▇▇█▇████████████
wandb:      bert/epoch ▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
wandb:       bert/loss █▇▆▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
wandb: bert/val_ndcg10 ▁▃▄▅▅▆▇██████████████▇█
wandb: 
wandb: Run summary:
wandb:        bert/acc 0.67131
wandb:      bert/epoch 23
wandb:       bert/loss 7.30363
wandb: bert/val_ndcg10 0.03833
wandb: 


wandb:  View run nb-bert-retrain at: https://wandb.ai/lamgiang-fpt-university/bert4rec-longterm-rerank/runs/rv4j9phm
wandb:  View project at: https://wandb.ai/lamgiang-fpt-university/bert4rec-longterm-rerank
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260619_125145-rv4j9phm\logs


saved bert4rec_longterm.pth | best VAL NDCG@10 = 0.04


## 4. Đánh giá BERT trên TEST (graded NDCG / Recall / MeanRating)

In [5]:
res_bert = R.eval_bert(bert, users, DEVICE, ks=(5,10,20))
json.dump(res_bert, open(os.path.join(HERE,"results_bert.json"),"w"), ensure_ascii=False, indent=2)
print(json.dumps(res_bert, ensure_ascii=False, indent=2))

{
  "NDCG@5": 0.0344728411532718,
  "Recall@5": 0.011953588171009019,
  "NDCG@10": 0.03694276721597422,
  "Recall@10": 0.022047979062496392,
  "NDCG@20": 0.048731478363197855,
  "Recall@20": 0.0473534095949075,
  "MeanRating@10": 1.2210292494481236
}
